# Document Ingester
> Chunk and ingest documents into context items for the Anchor pipeline.

`DocumentIngester` takes raw text, runs it through a chunker, and produces
a list of `ContextItem` objects annotated with source type, priority, and
user-supplied metadata.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.ingestion import DocumentIngester
from anchor.ingestion.chunkers import RecursiveCharacterChunker
from anchor.models import SourceType

## Configure the Chunker
`RecursiveCharacterChunker` splits text using a hierarchy of separators
(paragraphs, sentences, words) to produce chunks that respect natural
boundaries.

In [ ]:
chunker = RecursiveCharacterChunker(chunk_size=256, overlap=50)

print(f"Chunk size: {chunker.chunk_size}")
print(f"Overlap:    {chunker.overlap}")

## Create the Ingester
`source_type` tags every item produced by this ingester. `priority` controls
where items land in the context window (lower = closer to the system prompt).

In [ ]:
ingester = DocumentIngester(
    chunker=chunker,
    source_type=SourceType.RETRIEVAL,
    priority=5,
)

print(f"Source type: {ingester.source_type}")
print(f"Priority:    {ingester.priority}")

## Prepare a Sample Document
We use a multi-paragraph mock document so the chunker has enough text
to produce several chunks.

In [ ]:
text = (
    "Anchor is a framework for building context-aware AI applications. "
    "It provides tools for memory management, retrieval-augmented generation, "
    "and intelligent context assembly. The framework is designed to be modular, "
    "allowing developers to pick and choose the components they need.\n\n"
    "The ingestion module handles converting raw documents into structured "
    "context items. It supports multiple chunking strategies, each optimized "
    "for different content types. Text is split into overlapping chunks to "
    "preserve context at boundaries.\n\n"
    "Once ingested, context items flow through the pipeline where they are "
    "scored, filtered, and assembled into the final prompt. Priority values "
    "determine ordering, while token budgets control how much context fits "
    "into the model's context window.\n\n"
    "Anchor supports various source types including retrieval results, "
    "conversation history, tool outputs, and system instructions. Each source "
    "type can be configured with different priority levels and token budgets "
    "to ensure the most relevant information reaches the model."
)

print(f"Document length: {len(text)} characters")

## Ingest the Document
`ingest_text()` chunks the text and wraps each chunk in a `ContextItem`.
Pass optional `metadata` to attach to every item.

In [ ]:
items = ingester.ingest_text(text, metadata={"source": "readme.md"})

print(f"Generated {len(items)} context items\n")
for i, item in enumerate(items):
    content_preview = str(item.content)[:70]
    print(f"  [{i}] priority={item.priority}  "
          f"type={item.source_type.value}")
    print(f"      {content_preview}...")
    print()

## Inspect Metadata
Each context item carries the metadata you passed to `ingest_text()`.

In [ ]:
first_item = items[0]

print(f"Content length: {len(str(first_item.content))} chars")
print(f"Source type:    {first_item.source_type}")
print(f"Priority:       {first_item.priority}")
print(f"Metadata:       {first_item.metadata}")

## Ingest Multiple Documents
Call `ingest_text()` for each document. Metadata lets you track provenance.

In [ ]:
docs = [
    ("First document about Python basics and syntax.", {"source": "python.md"}),
    ("Second document covering JavaScript and web APIs.", {"source": "js.md"}),
    ("Third document on database design and SQL patterns.", {"source": "sql.md"}),
]

all_items = []
for content, meta in docs:
    batch = ingester.ingest_text(content, metadata=meta)
    all_items.extend(batch)
    print(f"  {meta['source']}: {len(batch)} items")

print(f"\nTotal items across all documents: {len(all_items)}")

## Key Takeaways
- `DocumentIngester` pairs a chunker with source-type and priority settings
- `ingest_text()` returns a list of `ContextItem` objects ready for the pipeline
- Metadata flows through to every item for provenance tracking
- Swap chunkers to change how documents are split without changing ingestion logic

**Next:** [Chunking Strategies](02_chunking_strategies.ipynb)